### Incremental Data Loading

In [0]:
CREATE DATABASE sales_new;


In [0]:
CREATE TABLE sales_new.Orders (
    OrderID INT,
    OrderDate DATE,
    CustomerID INT,
    CustomerName VARCHAR(100),
    CustomerEmail VARCHAR(100),
    ProductID INT,
    ProductName VARCHAR(100),
    ProductCategory VARCHAR(50),
    RegionID INT,
    RegionName VARCHAR(50),
    Country VARCHAR(50),
    Quantity INT,
    UnitPrice DECIMAL(10,2),
    TotalAmount DECIMAL(10,2)
);

In [0]:
INSERT INTO sales_new.Orders (OrderID, OrderDate, CustomerID, CustomerName, CustomerEmail, ProductID, ProductName, ProductCategory, RegionID, RegionName, Country, Quantity, UnitPrice, TotalAmount) 
VALUES 
(1, '2024-02-01', 101, 'Alice Johnson', 'alice@example.com', 201, 'Laptop', 'Electronics', 301, 'North America', 'USA', 2, 800.00, 1600.00),
(2, '2024-02-02', 102, 'Bob Smith', 'bob@example.com', 202, 'Smartphone', 'Electronics', 302, 'Europe', 'Germany', 1, 500.00, 500.00),
(3, '2024-02-03', 103, 'Charlie Brown', 'charlie@example.com', 203, 'Tablet', 'Electronics', 303, 'Asia', 'India', 3, 300.00, 900.00),
(4, '2024-02-04', 101, 'Alice Johnson', 'alice@example.com', 204, 'Headphones', 'Accessories', 301, 'North America', 'USA', 1, 150.00, 150.00),
(5, '2024-02-05', 104, 'David Lee', 'david@example.com', 205, 'Gaming Console', 'Electronics', 302, 'Europe', 'France', 1, 400.00, 400.00),
(6, '2024-02-06', 102, 'Bob Smith', 'bob@example.com', 206, 'Smartwatch', 'Electronics', 303, 'Asia', 'China', 2, 200.00, 400.00),
(7, '2024-02-07', 105, 'Eve Adams', 'eve@example.com', 201, 'Laptop', 'Electronics', 301, 'North America', 'Canada', 1, 800.00, 800.00),
(8, '2024-02-08', 106, 'Frank Miller', 'frank@example.com', 207, 'Monitor', 'Accessories', 302, 'Europe', 'Italy', 2, 250.00, 500.00),
(9, '2024-02-09', 107, 'Grace White', 'grace@example.com', 208, 'Keyboard', 'Accessories', 303, 'Asia', 'Japan', 3, 100.00, 300.00),
(10, '2024-02-10', 104, 'David Lee', 'david@example.com', 209, 'Mouse', 'Accessories', 301, 'North America', 'USA', 1, 50.00, 50.00);


In [0]:
-- INSERT INTO sales_new.Orders (OrderID, OrderDate, CustomerID, CustomerName, CustomerEmail, ProductID, ProductName, ProductCategory, RegionID, RegionName, Country, Quantity, UnitPrice, TotalAmount) 
-- VALUES 
-- (11, '2024-02-11', 101, 'Alice Johnson', 'alice@example.com', 201, 'Laptop', 'Electronics', 301, 'North America', 'USA', 2, 800.00, 1600.00),
-- (12, '2024-02-12', 102, 'Bob Smith', 'bob@example.com', 202, 'Smartphone', 'Electronics', 302, 'Europe', 'Germany', 1, 500.00, 500.00),
-- (13, '2024-02-13', 103, 'Charlie Brown', 'charlie@example.com', 203, 'Tablet', 'Electronics', 303, 'Asia', 'India', 3, 300.00, 900.00),
-- (14, '2024-02-14', 101, 'Alice Johnson', 'alice@example.com', 204, 'Headphones', 'Accessories', 301, 'North America', 'USA', 1, 150.00, 150.00),
-- (15, '2024-02-15', 104, 'David Lee', 'david@example.com', 205, 'Gaming Console', 'Electronics', 302, 'Europe', 'France', 1, 400.00, 400.00)

In [0]:
select * from sales_new.Orders;

  # Data Warehousing

In [0]:
CREATE DATABASE ordersDWH

### Staging Layer

In [0]:
-- Initial load
CREATE OR REPLACE TABLE ordersDWH.stg_sales
AS
SELECT * FROM sales_new.orders

### Transformation

In [0]:
  CREATE VIEW ordersDWH.trans_sales
  AS
  SELECT * from ordersDWH.stg_sales WHERE Quantity is not null 

In [0]:
SELECT * FROM ordersDWH.trans_sales

### Core Layer

#### DimCustomer

In [0]:
CREATE OR REPLACE TABLE ordersDWH.DimCustomer
(
  CustomerID INT,
  CustomerName STRING,
  CustomerEmail STRING,
  DimCustomerKey INT
)

In [0]:
CREATE OR REPLACE VIEW ordersDWH.view_DimCustomers
AS

SELECT T.*,row_number() over(ORDER BY T.CustomerID) as DimCustomerKey FROM
(SELECT 
  DISTINCT(CustomerID) as CustomerID,
  CustomerName,
  CustomerEmail
FROM 
  ordersDWH.trans_sales) AS T


In [0]:
INSERT INTO ordersDWH.DimCustomer 
SELECT * from ordersDWH.view_DimCustomers

In [0]:
SELECT * FROM ordersDWH.DimCustomer

#### DimProducts

In [0]:
CREATE TABLE ordersDWH.DimProducts(
  ProductID INT,
  ProductName STRING,
  ProductCategory STRING,
  DimProductKey INT
)

In [0]:
CREATE VIEW ordersDWH.view_DimProducts
AS

SELECT V.*, row_number() over(order by V.ProductID) AS DimProductKey
FROM
(SELECT 
  DISTINCT(ProductID) AS ProductID, 
  ProductName, 
  ProductCategory
FROM 
  ordersDWH.trans_sales) AS V

In [0]:
INSERT INTO ordersDWH.DimProducts
SELECT * FROM ordersDWH.view_DimProducts

#### DimRegion

In [0]:
CREATE TABLE ordersDWH.DimRegions
(
  RegionID INT,
  RegionName STRING,
  Country STRING,
  DimRegionKey INT
)

In [0]:
CREATE VIEW ordersDWH.view_DimRegions
AS
SELECT S.*, row_number() over(order by S.RegionID) as DimRegionKey FROM
(SELECT Distinct(RegionID), RegionName, Country FROM ordersDWH.trans_sales) AS S

In [0]:
INSERT INTO ordersDWH.DimRegions 
SELECT * FROM ordersDWH.view_DimRegions


#### DimDate

In [0]:
CREATE OR REPLACE TABLE ordersDWH.DimDate
(
  OrderDate DATE,
  DimDateKey INT
)

In [0]:
CREATE VIEW ordersDWH.view_DimDate
AS

SELECT T.*, row_number() over(order by T.orderDate) FROM 
(SELECT DISTINCT(OrderDate) AS OrderDate FROM ordersDWH.trans_sales) AS T

In [0]:
INSERT INTO ordersDWH.DimDate 
SELECT * FROM ordersDWH.view_DimDate

In [0]:
SELECT * FROM ordersDWH.DimDate

### Fact Table

In [0]:
SELECT * FROM ordersDWH.trans_sales

In [0]:
CREATE OR REPLACE TABLE ordersDWH.FactSales
( 
  OrderID INT,
  Quantity INT,
  UnitPrice DECIMAL,
  TotalAmount DECIMAL,
  DimProductKey INT,
  DimCustomerKey INT,
  DimRegionKey INT,
  DimDateKey INT
)

In [0]:
SELECT 
  F.OrderID,
  F.Quantity,
  F.UnitPrice,
  F.TotalAmount,
  DC.DimCustomerKey,
  DP.DimProductKey,
  DD.DimDateKey,
  DR.DimRegionKey
FROM 
  ordersDWH.trans_sales F 
LEFT JOIN
  ordersDWH.dimcustomer DC
ON 
  F.CustomerID = DC.CustomerID
LEFT JOIN 
  ordersDWH.DimProducts DP
ON 
  F.ProductID = DP.ProductID
LEFT JOIN 
  ordersDWH.DimDate DD
ON 
  F.OrderDate = DD.OrderDate
LEFT JOIN 
  ordersDWH.DimRegions DR
ON 
  F.Country = DR.Country
